# Wellness Scoring Models

This notebook trains 4 ML models to predict wellness scores from wearable data:
1. **Sleep Quality** (0-100)
2. **Stress/Recovery** (0-100)
3. **Activity/Strain** (0-100)
4. **Illness Risk** (0-100)

## Data Source
Processed Parquet files from the Glue ETL pipeline in S3.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import boto3
import io

PROCESSED_BUCKET = 'health-pipeline-processed-939295406035'
CURATED_BUCKET = 'health-pipeline-curated-939295406035'
REGION = 'us-east-1'

## 1. Load Processed Data

In [ ]:
# Read Parquet from S3
df = pd.read_parquet(f's3://{PROCESSED_BUCKET}/daily_summaries/', storage_options={'region_name': REGION})
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
df.head()

In [ ]:
# Inspect available columns
print('Columns:', df.columns.tolist())
print('\nNull counts:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

## 2. Feature Engineering

Create composite wellness scores as targets using domain knowledge:
- Sleep Quality: sleep_duration, sleep_efficiency, deep_sleep_minutes, rem_sleep_minutes
- Stress/Recovery: hrv_rmssd, resting_hr, stress_score
- Activity/Strain: steps, active_minutes, calories
- Illness Risk: spo2_avg, skin_temp_deviation, resting_hr elevation

In [ ]:
def compute_sleep_quality(row):
    """Composite sleep score based on duration, efficiency, and deep sleep."""
    duration_score = np.clip(row.get('sleep_duration', 0) / 480 * 100, 0, 100)  # 8hr = 100
    efficiency_score = np.clip(row.get('sleep_efficiency', 0) * 100, 0, 100)
    deep_score = np.clip(row.get('deep_sleep_minutes', 0) / 90 * 100, 0, 100)  # 90min = 100
    return np.clip(0.4 * duration_score + 0.3 * efficiency_score + 0.3 * deep_score, 0, 100)


def compute_stress_recovery(row):
    """Lower stress = higher recovery score. Based on HRV and resting HR."""
    hrv_score = np.clip(row.get('hrv_rmssd', 30) / 80 * 100, 0, 100)  # 80ms = excellent
    hr_score = np.clip((80 - row.get('resting_hr', 70)) / 20 * 100, 0, 100)  # lower = better
    stress_inv = np.clip(100 - row.get('stress_score', 50), 0, 100)
    return np.clip(0.4 * hrv_score + 0.3 * hr_score + 0.3 * stress_inv, 0, 100)


def compute_activity_strain(row):
    """Activity intensity score based on steps and active minutes."""
    steps_score = np.clip(row.get('steps', 0) / 10000 * 100, 0, 100)
    active_score = np.clip(row.get('active_minutes', 0) / 60 * 100, 0, 100)  # 60min = 100
    return np.clip(0.5 * steps_score + 0.5 * active_score, 0, 100)


def compute_illness_risk(row):
    """Higher score = higher risk. Based on SpO2 dip, temp deviation, elevated HR."""
    spo2_risk = np.clip((97 - row.get('spo2_avg', 97)) / 3 * 100, 0, 100)  # below 97 = risk
    temp_risk = np.clip(abs(row.get('skin_temp_deviation', 0)) / 2 * 100, 0, 100)
    hr_risk = np.clip((row.get('resting_hr', 60) - 60) / 20 * 100, 0, 100)  # elevated = risk
    return np.clip(0.4 * spo2_risk + 0.3 * temp_risk + 0.3 * hr_risk, 0, 100)

In [ ]:
# Compute target scores
df['sleep_quality_score'] = df.apply(compute_sleep_quality, axis=1)
df['stress_recovery_score'] = df.apply(compute_stress_recovery, axis=1)
df['activity_strain_score'] = df.apply(compute_activity_strain, axis=1)
df['illness_risk_score'] = df.apply(compute_illness_risk, axis=1)

print('Score distributions:')
df[['sleep_quality_score', 'stress_recovery_score', 'activity_strain_score', 'illness_risk_score']].describe()

## 3. Model Training

In [ ]:
# Select numeric features (exclude targets and metadata)
target_cols = ['sleep_quality_score', 'stress_recovery_score', 'activity_strain_score', 'illness_risk_score']
meta_cols = ['date', 'participant_id', 'id', 'year', 'month']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in target_cols + meta_cols]

print(f'Using {len(feature_cols)} features: {feature_cols[:10]}...')

# Drop rows with too many nulls
df_model = df[feature_cols + target_cols].dropna(thresh=len(feature_cols) * 0.5)
df_model = df_model.fillna(df_model.median())
print(f'Training data: {len(df_model)} rows')

In [ ]:
results = {}

for target in target_cols:
    print(f'\n--- Training: {target} ---')
    X = df_model[feature_cols]
    y = df_model[target]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[target] = {'model': model, 'mae': mae, 'r2': r2}
    print(f'  MAE: {mae:.2f}, R²: {r2:.3f}')

print('\n=== Summary ===')
for name, r in results.items():
    print(f'{name}: MAE={r["mae"]:.2f}, R²={r["r2"]:.3f}')

## 4. Feature Importance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, r) in zip(axes.flat, results.items()):
    importances = r['model'].feature_importances_
    top_idx = np.argsort(importances)[-10:]
    ax.barh([feature_cols[i] for i in top_idx], importances[top_idx])
    ax.set_title(name.replace('_score', '').replace('_', ' ').title())

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()

## 5. Batch Inference — Write Scores to Curated Bucket

In [ ]:
# Predict scores for all data
X_all = df_model[feature_cols]
for target in target_cols:
    pred_col = target.replace('_score', '_predicted')
    df_model[pred_col] = results[target]['model'].predict(X_all)

# Write predictions to curated bucket
output_cols = ['participant_id', 'date'] if 'participant_id' in df_model.columns else []
output_cols += [t.replace('_score', '_predicted') for t in target_cols]

# Use available ID columns
score_df = df_model[[c for c in output_cols if c in df_model.columns]].copy()
score_df.to_parquet(
    f's3://{CURATED_BUCKET}/wellness_scores/scores.parquet',
    index=False,
    storage_options={'region_name': REGION},
)
print(f'Wrote {len(score_df)} predictions to s3://{CURATED_BUCKET}/wellness_scores/')

## 6. Model Evaluation Summary

| Score | MAE | R² | Top Features |
|-------|-----|----|--------------|
| Sleep Quality | - | - | sleep_duration, deep_sleep_minutes, sleep_efficiency |
| Stress/Recovery | - | - | hrv_rmssd, resting_hr, stress_score |
| Activity/Strain | - | - | steps, active_minutes, calories |
| Illness Risk | - | - | spo2_avg, skin_temp_deviation, resting_hr |